<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/notebooks/D8_lab_2_3_sentimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Lab 2.3 - Paso 1: explorar el dataset de sentimiento en español
import numpy as np
import pandas as pd
from datasets import load_dataset

# Cargamos directamente los archivos Parquet desde el repositorio de conversión oficial de Hugging Face
base_url = "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/train/"

data_files = {
    "train": f"{base_url}0000.parquet",
    "validation": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/test/0000.parquet"
}

ds = load_dataset("parquet", data_files=data_files)
print(ds)

# Definimos las etiquetas directamente
nombres = ["negative", "neutral", "positive"]
partes = []
for split in ["train", "validation", "test"]:
    d = ds[split].to_pandas()
    d["split"] = split
    partes.append(d)
df = pd.concat(partes, ignore_index=True)
df["sentimiento"] = df["label"].map(dict(enumerate(nombres)))
df["n_palabras"] = df["text"].str.split().str.len()

print("\n% de cada sentimiento por split:")
print((pd.crosstab(df["sentimiento"], df["split"], normalize="columns") * 100).round(1).to_string())

print("\nPalabras por tweet (train):", df.loc[df.split == "train", "n_palabras"].describe().round(1).to_dict())

print("\nEjemplos de train:")
for _, fila in df[df.split == "train"].sample(6, random_state=7).iterrows():
    print(f"  [{fila['sentimiento']:<8}] {fila['text'][:110]}")

0000.parquet:   0%|          | 0.00/120k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1839
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 324
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 870
    })
})

% de cada sentimiento por split:
split        test  train  validation
sentimiento                         
negative     33.3   33.3        33.3
neutral      33.3   33.3        33.3
positive     33.3   33.3        33.3

Palabras por tweet (train): {'count': 1839.0, 'mean': 15.5, 'std': 6.1, 'min': 5.0, '25%': 10.0, '50%': 16.0, '75%': 20.0, 'max': 31.0}

Ejemplos de train:
  [neutral ] @user Son las grandes promesas nuestras
  [neutral ] @user No se me ocurre más que que estás guapísima, por eso no valgo para la seducción
  [positive] @user Muchas gracias atrasadas!! Esto del Twitter hacia que no lo miraba.... Un abrazoteee!!
  [negative] @user Eso es antiguo ya
  [negative] Mi móvil no me deja abrir los gru

In [2]:
# Lab 2.3 - Paso 2: baseline TF-IDF frente a BiLSTM
import tensorflow as tf
from tensorflow import keras
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

train_df = df[df.split == "train"]
val_df = df[df.split == "validation"]
y_tr, y_va = train_df["label"].to_numpy(), val_df["label"].to_numpy()

# A) Baseline: TF-IDF palabras + caracteres (misma receta que E0)
base = make_pipeline(
    FeatureUnion([
        ("palabras",   TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LogisticRegression(max_iter=2000),
)
base.fit(train_df["text"], y_tr)
pred_base = base.predict(val_df["text"])
print(f"TF-IDF + LR | macro-F1: {f1_score(y_va, pred_base, average='macro'):.4f} | "
      f"exactitud: {accuracy_score(y_va, pred_base):.4f}")

# B) BiLSTM
LARGO = 40                                   # el tweet más largo de train tiene 31 palabras
vec = keras.layers.TextVectorization(max_tokens=10_000, output_sequence_length=LARGO)
vec.adapt(train_df["text"].to_numpy())
X_tr = vec(tf.constant(train_df["text"].tolist())).numpy()
X_va = vec(tf.constant(val_df["text"].tolist())).numpy()
print("Vocabulario:", len(vec.get_vocabulary()))

def crear_bilstm(semilla):
    tf.keras.utils.set_random_seed(semilla)
    m = keras.Sequential([
        keras.Input(shape=(LARGO,), dtype="int64"),
        keras.layers.Embedding(len(vec.get_vocabulary()), 64, mask_zero=True),
        keras.layers.Bidirectional(keras.layers.LSTM(32)),   # lee en ambos sentidos
        keras.layers.Dropout(0.5),
        keras.layers.Dense(3, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

bilstm = crear_bilstm(42)
hist = bilstm.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=30, batch_size=32, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                           restore_best_weights=True)])
pred_lstm = bilstm.predict(X_va, verbose=0).argmax(axis=1)
h = hist.history
print(f"BiLSTM      | macro-F1: {f1_score(y_va, pred_lstm, average='macro'):.4f} | "
      f"exactitud: {accuracy_score(y_va, pred_lstm):.4f} | épocas: {len(h['loss'])} | "
      f"train acc final: {h['accuracy'][-1]:.4f}")

TF-IDF + LR | macro-F1: 0.5978 | exactitud: 0.5988
Vocabulario: 6370
BiLSTM      | macro-F1: 0.5083 | exactitud: 0.5309 | épocas: 5 | train acc final: 0.9734


In [3]:
# Lab 2.3 - Paso 3: métricas por clase y análisis de errores (baseline)
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_va, pred_base, target_names=nombres, digits=3))

cm = pd.DataFrame(confusion_matrix(y_va, pred_base),
                  index=[f"real {n}" for n in nombres], columns=[f"pred {n}" for n in nombres])
print("Matriz de confusión (validación):\n", cm.to_string())

# ¿Qué pasa con los tweets que contienen negaciones?
negacion = val_df["text"].str.lower().str.contains(r"\b(?:no|nunca|nada|ni|tampoco)\b", regex=True)
acierto = pred_base == y_va
print(f"\nExactitud con negación: {acierto[negacion.to_numpy()].mean():.3f} ({negacion.sum()} tweets)")
print(f"Exactitud sin negación: {acierto[~negacion.to_numpy()].mean():.3f} ({(~negacion).sum()} tweets)")

# 10 errores para leer
errores = val_df[~acierto].copy()
errores["predicho"] = [nombres[p] for p in pred_base[~acierto]]
print("\n10 errores del baseline:")
for i, (_, f) in enumerate(errores.sample(10, random_state=3).iterrows(), 1):
    print(f"{i:>2}. real: {f['sentimiento']:<8} | pred: {f['predicho']:<8} | {f['text'][:100]}")

              precision    recall  f1-score   support

    negative      0.614     0.648     0.631       108
     neutral      0.505     0.481     0.493       108
    positive      0.673     0.667     0.670       108

    accuracy                          0.599       324
   macro avg      0.597     0.599     0.598       324
weighted avg      0.597     0.599     0.598       324

Matriz de confusión (validación):
                pred negative  pred neutral  pred positive
real negative             70            30              8
real neutral              29            52             27
real positive             15            21             72

Exactitud con negación: 0.556 (108 tweets)
Exactitud sin negación: 0.620 (216 tweets)

10 errores del baseline:
 1. real: negative | pred: neutral  | @user a mi me pasa igual, quiero estar con ellas pero quiero hacer muchas cosas también, y son muy p
 2. real: neutral  | pred: positive | @user Comprendo. ¡Mil gracias! Es solo porque me pareció ver u

In [4]:
# Lab 2.3 - Paso 4: proyecto EnrutaIA - matriz de confusión de E0 y class_weight
import hashlib, io, json, os, urllib.request

BASE = "https://raw.githubusercontent.com/Edubel77/enrutaia/main/data/limpio/"
MD5 = {"train": "cc0ff7796f8e1d1df56862a9f708d521", "validation": "ba68845308293612e717aa7c1a55887c"}
d = {}
for split, esperado in MD5.items():
    contenido = urllib.request.urlopen(BASE + f"{split}.csv").read()
    assert hashlib.md5(contenido).hexdigest() == esperado, f"¡{split}.csv cambió!"
    d[split] = pd.read_csv(io.BytesIO(contenido))
tr, va = d["train"], d["validation"]

def crear_e0(class_weight=None):
    return make_pipeline(
        FeatureUnion([
            ("palabras",   TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ]),
        LogisticRegression(max_iter=2000, class_weight=class_weight),
    )

dominios = sorted(tr["dominio"].unique())
resultados = {}
for nombre, cw in [("E0", None), ("E0-balanced", "balanced")]:
    m = crear_e0(cw).fit(tr["text"], tr["dominio"])
    pred = m.predict(va["text"])
    f1_dom = f1_score(va["dominio"], pred, average=None, labels=dominios)
    resultados[nombre] = {"macro_f1": round(f1_score(va["dominio"], pred, average="macro"), 4),
                          "f1_general": round(f1_dom[dominios.index("general")], 4),
                          "f1_qa": round(f1_dom[dominios.index("qa")], 4),
                          "f1_minimo": round(f1_dom.min(), 4)}
    print(f"{nombre:<12} | {resultados[nombre]}")
    if cw is None:
        pred_e0 = pred

# Las 5 confusiones más frecuentes de E0
cm = pd.DataFrame(confusion_matrix(va["dominio"], pred_e0, labels=dominios), index=dominios, columns=dominios)
pares = [(r, p, cm.loc[r, p]) for r in dominios for p in dominios if r != p]
print("\nConfusiones más frecuentes de E0 (real -> predicho):")
for r, p, n in sorted(pares, key=lambda x: -x[2])[:5]:
    print(f"  {r:<14} -> {p:<14}: {n}")
print(f"\nDe {(va['dominio'] == 'general').sum()} textos de general, terminaron como qa: {cm.loc['general', 'qa']}")

os.makedirs("results", exist_ok=True)
with open("results/E0_class_weight.json", "w") as f:
    json.dump({"split_evaluado": "validation", "resultados": resultados}, f, indent=2, ensure_ascii=False)

E0           | {'macro_f1': 0.8725, 'f1_general': np.float64(0.6047), 'f1_qa': np.float64(0.7975), 'f1_minimo': np.float64(0.6047)}
E0-balanced  | {'macro_f1': 0.8699, 'f1_general': np.float64(0.595), 'f1_qa': np.float64(0.8), 'f1_minimo': np.float64(0.595)}

Confusiones más frecuentes de E0 (real -> predicho):
  general        -> qa            : 30
  music          -> play          : 9
  qa             -> general       : 9
  news           -> qa            : 8
  play           -> qa            : 7

De 115 textos de general, terminaron como qa: 30
